# Supervised Regression

Trains a regression model on tabular survey columns. Handles automatic preprocessing, SHAP feature importance, cross-validation, residual diagnostics, and saves predicted values back to SuAVE as a new #number column.

**Models (scikit-learn):** Linear Regression, Ridge, Lasso, ElasticNet, Random Forest, Gradient Boosting, XGBoost, AdaBoost, SVR, K-Nearest Neighbors.  
**Statistical model:** OLS with p-values, confidence intervals, F-statistic, AIC/BIC (statsmodels).

Run the five parameter cells first, then work through Steps 1-5.

In [ ]:
# -- Colab bootstrap (no-op on Binder / JupyterHub) ------------------------
import os, sys
if "google.colab" in sys.modules:
    import subprocess
    if os.path.isdir("/tmp/suave-nb/.git"):
        subprocess.run(["git","-C","/tmp/suave-nb","fetch","--depth=1","origin","main"],
                       capture_output=True)
        subprocess.run(["git","-C","/tmp/suave-nb","reset","--hard","origin/main"],
                       capture_output=True)
    else:
        _r = subprocess.run(
            ["git","clone","--depth=1",
             "https://github.com/izaslavsky/suave-notebooks.git","/tmp/suave-nb"],
            capture_output=True, text=True)
        if _r.returncode != 0:
            raise RuntimeError(f"Could not clone suave-notebooks:\n{_r.stderr}")
    sys.path.insert(0, "/tmp/suave-nb/helpers")


In [ ]:
# -- Colab only: mount Google Drive to load session credentials ------------
import sys
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')


In [ ]:
# -- Load SuAVE parameters -------------------------------------------------
import sys, os, requests as _req, urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
from urllib.parse import urlparse as _urlparse
sys.path.insert(0, '../../helpers')
import suave_utils as su
from IPython.display import display, Markdown
import pandas as pd

def printmd(string):
    display(Markdown(string))

if not su.ENV.colab:
    _p = su.load_params()
    if _p is None:
        raise RuntimeError('No SuAVE params. Open via SuAVEDispatch.ipynb first.')
else:
    _p = su.load_params(_silent=True)


In [ ]:
# -- Colab: enter credentials if Drive had no session ----------------------
if su.ENV.colab:
    import getpass as _gp
    if _p is not None:
        _survey = _p.get('survey', '?')
        display(HTML(
            f'<p style="font-size:12px;margin:3px 0">Session active for survey '
            f'<b>{_survey}</b>. Press Enter to keep it, or paste a new '
            f'SUAVE_TOKEN to switch surveys.</p>'))
    else:
        display(HTML('<p style="color:#e07000;font-size:12px">No session loaded. '
                     'Enter credentials from SuAVEDispatch.</p>'))
    _tok = _gp.getpass('SUAVE_TOKEN (Enter to keep current): ')
    if _tok.strip():
        _host = input('SUAVE_HOST (e.g. george.tirebiter.org): ')
        _p = su.load_params(token=_tok.strip(), host=_host.strip())
    elif _p is None:
        raise RuntimeError(
            'No session and no token provided. '
            'Run SuAVEDispatch first, or enter SUAVE_TOKEN above.')
else:
    su._skipped('Colab only')


In [ ]:
# -- Variables used throughout this notebook --------------------------------
if _p is None:
    raise RuntimeError('Parameters not loaded. Run the cell above to enter credentials.')
user          = _p.get('user', '')
survey_url    = _p.get('surveyurl', '')
csv_file      = _p.get('csv', '')
dzc_file      = _p.get('dzc', '')
active_object = _p.get('activeobject', 'none')
_caps         = su.detect_capabilities(_p)
views         = ','.join(_caps.get('views', []))
view          = 'grid'
localdzc             = _caps.get('localdzc', '')
full_images          = _caps.get('full_images', '')
full_images_location = full_images

absolutePath = os.path.expanduser('~/temp_csvs/')
os.makedirs(absolutePath, exist_ok=True)
_origin  = _urlparse(survey_url)
_csv_url = f"{_origin.scheme}://{_origin.netloc}/surveys/{csv_file}"
_r = _req.get(_csv_url, timeout=30, verify=False)
_r.raise_for_status()
with open(absolutePath + csv_file, 'wb') as _f:
    _f.write(_r.content)
su.show_params(_p)


In [ ]:
import sys, subprocess
sys.path.insert(1, '../../helpers')
import panel_libs as panellibs
import suave_integration as suaveint
from collections import OrderedDict as _OD

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML, Markdown, clear_output
import ipywidgets as widgets
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     KFold, cross_val_score)
from sklearn.preprocessing import LabelEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, classification_report,
    r2_score, mean_squared_error, mean_absolute_error)
from sklearn.linear_model import (LogisticRegression, Ridge, Lasso,
                                   ElasticNet, LinearRegression)
from sklearn.ensemble import (RandomForestClassifier, RandomForestRegressor,
    GradientBoostingClassifier, GradientBoostingRegressor,
    AdaBoostClassifier, AdaBoostRegressor,
    ExtraTreesClassifier, ExtraTreesRegressor)
from sklearn.svm import SVC, SVR
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
import statsmodels.api as sm

for _pkg in ('xgboost', 'shap'):
    try:
        __import__(_pkg)
    except ImportError:
        print(f'Installing {_pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', _pkg])
from xgboost import XGBClassifier, XGBRegressor
import shap

printmd('**Imports complete.**')


In [ ]:
df_raw = panellibs.extract_data(absolutePath + csv_file)
print(f"Survey: {csv_file}  |  {len(df_raw):,} rows  |  {len(df_raw.columns)} columns")

_SKIP = ('#img', '#link', '#href', '#netvis', '#hidden', '#hiddenmore',
         '#name', '#lat', '#lon', '#coord', '#textlocation')

def _usable(c):   return not any(q in c for q in _SKIP)
def _isnumcol(c): return '#number' in c
def _lbl(c):      return c.split('#')[0].strip() or c

usable   = [c for c in df_raw.columns if _usable(c)]
num_raw  = [c for c in usable if _isnumcol(c)]
cat_raw  = [c for c in usable if not _isnumcol(c)]
_lbl2raw = {_lbl(c): c for c in usable}
_labels  = list(_lbl2raw.keys())

rows = []
for c in usable:
    rows.append({'Column': _lbl(c),
                 'Type': 'numeric' if _isnumcol(c) else 'categorical',
                 'Unique values': df_raw[c].nunique(),
                 'Missing': df_raw[c].isnull().sum()})
display(pd.DataFrame(rows)
        .style.set_properties(**{'font-size': '11px'})
        .set_caption(f'{len(usable)} usable columns  '
                     f'({len(num_raw)} numeric, {len(cat_raw)} categorical)'))


In [ ]:
# -- Step 1: select target, features, and split ----------------------------
num_labels = [_lbl(c) for c in num_raw]
y_dd = widgets.Dropdown(
    options=num_labels if num_labels else _labels,
    description='Target y:', style={'description_width': '100px'},
    layout=widgets.Layout(width='380px'))
X_ms = widgets.SelectMultiple(
    options=_labels, value=_labels[1:min(7, len(_labels))],
    description='Features X:', rows=min(12, len(_labels)),
    style={'description_width': '100px'},
    layout=widgets.Layout(width='380px', height='210px'))
split_s = widgets.FloatSlider(
    value=0.2, min=0.05, max=0.5, step=0.05, description='Test set:',
    readout_format='.0%', style={'description_width': '100px'},
    layout=widgets.Layout(width='380px'))
seed_t = widgets.IntText(
    value=42, description='Seed:',
    style={'description_width': '100px'}, layout=widgets.Layout(width='220px'))
_y_info = widgets.HTML()

def _update_y_info(change):
    col  = _lbl2raw.get(y_dd.value)
    if col is None: return
    vals = pd.to_numeric(df_raw[col], errors='coerce').dropna()
    color = '#10b981' if _isnumcol(col) else '#e07000'
    note  = '{:,} numeric values -- range [{:.4g}, {:.4g}], mean {:.4g}'.format(
        len(vals), vals.min(), vals.max(), vals.mean())
    _y_info.value = (
        '<div style="font-size:12px;color:{};padding:3px 0">{}</div>'.format(color, note))

y_dd.observe(_update_y_info, names='value')
_update_y_info(None)

printmd('**Step 1 -- Choose a numeric target, features, and train/test split:**')
display(widgets.VBox([y_dd, _y_info, X_ms, split_s, seed_t]))


In [ ]:
_hp_widgets = {}

def _make_widget(pname, spec):
    W = widgets.Layout(width='440px')
    S = {'description_width': '160px'}
    kind = spec[0]
    if kind == 'flog':
        _, lo, hi, default, _desc = spec
        default = max(default, lo)
        return widgets.FloatLogSlider(
            value=default, base=10, min=np.log10(lo), max=np.log10(hi),
            step=0.05, description=pname + ':', readout_format='.3g',
            style=S, layout=W)
    elif kind == 'float':
        _, lo, hi, step, default, _desc = spec
        return widgets.FloatSlider(
            value=default, min=lo, max=hi, step=step,
            description=pname + ':', readout_format='.3f', style=S, layout=W)
    elif kind == 'int':
        _, lo, hi, default, _desc = spec
        return widgets.IntSlider(
            value=default, min=lo, max=hi,
            description=pname + ':', style=S, layout=W)
    elif kind == 'drop':
        _, opts, default, _desc = spec
        return widgets.Dropdown(
            options=[str(o) for o in opts], value=str(default),
            description=pname + ':', style=S,
            layout=widgets.Layout(width='440px'))
    return widgets.Label(pname)

_model_info = widgets.HTML()
_params_box  = widgets.VBox([])

def _on_model_change(change):
    name = model_dd.value
    _hp_widgets.clear()
    if name not in _CATALOG:
        _model_info.value = _SM_BLURBS.get(name, '')
        _params_box.children = []
        return
    spec   = _CATALOG[name]
    _url   = spec['url']
    _blurb = spec['blurb']
    _model_info.value = (
        '<div style="font-size:12px;padding:4px 0">'
        f'<a href="{_url}" target="_blank">{name} documentation</a> -- {_blurb}'
        '</div>')
    rows = []
    for pname, pspec in spec['params'].items():
        w = _make_widget(pname, pspec)
        _hp_widgets[pname] = w
        rows.append(w)
    _params_box.children = rows

def _hp_vals():
    out = {}
    for k, w in _hp_widgets.items():
        v = w.value
        if isinstance(v, str):
            if v == 'None':
                v = None
            else:
                try:    v = int(v)
                except ValueError:
                    try: v = float(v)
                    except ValueError: pass
        out[k] = v
    return out

# -- Step 2: select model and configure hyperparameters --------------------
_CATALOG = _OD([
  ('Linear Regression', dict(
    cls=LinearRegression, fit_kw=dict(),
    url='https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html',
    blurb='Ordinary least squares. No regularization -- can overfit when features are correlated or numerous.',
    shap='linear', params=_OD([]))),
  ('Ridge', dict(
    cls=Ridge, fit_kw=dict(),
    url='https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html',
    blurb='OLS with L2 regularization. Shrinks coefficients toward zero; stable with correlated features.',
    shap='linear',
    params=_OD([
      ('alpha', ('flog', 1e-3, 1e4, 1.0, 'Regularization strength (larger = more shrinkage)')),
    ]))),
  ('Lasso', dict(
    cls=Lasso, fit_kw=dict(max_iter=10000),
    url='https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html',
    blurb='OLS with L1 regularization. Sets some coefficients exactly to zero -- automatic feature selection.',
    shap='linear',
    params=_OD([
      ('alpha', ('flog', 1e-4, 100.0, 1.0, 'Regularization (larger = more sparsity)')),
    ]))),
  ('ElasticNet', dict(
    cls=ElasticNet, fit_kw=dict(max_iter=10000),
    url='https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ElasticNet.html',
    blurb='Combines L1 and L2 penalties. l1_ratio=1 behaves like Lasso; l1_ratio=0 like Ridge.',
    shap='linear',
    params=_OD([
      ('alpha',    ('flog', 1e-4, 100.0, 1.0, 'Overall regularization strength')),
      ('l1_ratio', ('float', 0.0, 1.0, 0.05, 0.5, 'L1 fraction (0=Ridge, 1=Lasso)')),
    ]))),
  ('Random Forest', dict(
    cls=RandomForestRegressor, fit_kw=dict(random_state=42, n_jobs=-1),
    url='https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html',
    blurb='Ensemble of regression trees. Captures nonlinear effects; robust to outliers.',
    shap='tree',
    params=_OD([
      ('n_estimators',      ('int',  50, 500, 100, 'Number of trees')),
      ('max_depth',         ('drop', [2,3,5,7,10,15,20,'None'], 'None', 'Max tree depth')),
      ('min_samples_split', ('int',   2,  20,   2, 'Min samples to split a node')),
    ]))),
  ('Gradient Boosting', dict(
    cls=GradientBoostingRegressor, fit_kw=dict(random_state=42),
    url='https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingRegressor.html',
    blurb='Sequential boosting of regression trees. Often the strongest predictor with careful tuning.',
    shap='tree',
    params=_OD([
      ('n_estimators',  ('int',   50, 500, 100, 'Number of boosting stages')),
      ('learning_rate', ('float', 0.01, 0.5, 0.05, 0.1, 'Shrinkage per tree')),
      ('max_depth',     ('drop',  [2,3,4,5,6,8], 3, 'Max depth per tree')),
      ('subsample',     ('float', 0.5, 1.0, 0.1, 1.0, 'Row fraction per tree')),
    ]))),
  ('XGBoost', dict(
    cls=XGBRegressor, fit_kw=dict(random_state=42, verbosity=0),
    url='https://xgboost.readthedocs.io/en/stable/parameter.html',
    blurb='Optimized gradient boosting with L1/L2 regularization and native missing-value handling.',
    shap='tree',
    params=_OD([
      ('n_estimators',  ('int',   50, 500, 100, 'Number of trees')),
      ('learning_rate', ('float', 0.01, 0.5, 0.05, 0.1, 'Step size shrinkage')),
      ('max_depth',     ('int',   2, 12, 6,  'Max depth per tree')),
      ('reg_alpha',     ('float', 0.0, 5.0, 0.5, 0.0, 'L1 weight regularization')),
      ('reg_lambda',    ('float', 0.0, 5.0, 0.5, 1.0, 'L2 weight regularization')),
    ]))),
  ('AdaBoost', dict(
    cls=AdaBoostRegressor, fit_kw=dict(random_state=42),
    url='https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.AdaBoostRegressor.html',
    blurb='Boosts regression by re-weighting high-residual samples on each round.',
    shap='tree',
    params=_OD([
      ('n_estimators',  ('int',   10, 300, 50, 'Number of weak learners')),
      ('learning_rate', ('float', 0.01, 2.0, 0.1, 1.0, 'Contribution per learner')),
    ]))),
  ('SVR', dict(
    cls=SVR, fit_kw=dict(),
    url='https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVR.html',
    blurb='Support Vector Regression. Effective for small-to-medium datasets; slow above ~10k rows.',
    shap='kernel',
    params=_OD([
      ('C',       ('flog', 1e-2, 1e3, 1.0, 'Regularization (larger = less tolerance for errors)')),
      ('epsilon', ('float', 0.0, 1.0, 0.05, 0.1, 'No-penalty tube width')),
      ('kernel',  ('drop', ['rbf','linear','poly'], 'rbf', 'Kernel type')),
    ]))),
  ('K-Nearest Neighbors', dict(
    cls=KNeighborsRegressor, fit_kw=dict(n_jobs=-1),
    url='https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsRegressor.html',
    blurb='Predicts by averaging k nearest training labels. Sensitive to feature scale; no training phase.',
    shap='kernel',
    params=_OD([
      ('n_neighbors', ('int',  1, 30, 5, 'Number of neighbors')),
      ('weights',     ('drop', ['uniform','distance'], 'uniform', 'Neighbor weighting')),
      ('metric',      ('drop', ['euclidean','manhattan','minkowski'], 'euclidean', 'Distance metric')),
    ]))),
])

_SM_OLS    = 'OLS (statsmodels)'
_SM_BLURBS = {
  _SM_OLS: (
    '<div style="font-size:12px;padding:4px 0">'
    '<a href="https://www.statsmodels.org/stable/generated/statsmodels.regression.linear_model.OLS.html"'
    ' target="_blank">statsmodels OLS documentation</a> -- '
    'Full statistical output: coefficient estimates, standard errors, t-statistics, '
    'p-values, confidence intervals, R2, adjusted R2, F-statistic, AIC, BIC.</div>'),
}

model_dd = widgets.Dropdown(
    options=list(_CATALOG.keys()) + [_SM_OLS],
    description='Model:', style={'description_width': '80px'},
    layout=widgets.Layout(width='380px'))
model_dd.observe(_on_model_change, names='value')
_on_model_change(None)

printmd('**Step 2 -- Select model and configure hyperparameters:**')
display(widgets.VBox([model_dd, _model_info, _params_box]))


In [ ]:
# -- Step 3: train and evaluate --------------------------------------------
y_col_raw  = _lbl2raw[y_dd.value]
X_col_raw  = [_lbl2raw[l] for l in X_ms.value if l != y_dd.value]
model_name = model_dd.value
test_size  = split_s.value
rand_seed  = seed_t.value

if not X_col_raw:
    raise ValueError('Select at least one feature column.')

df_model = df_raw[X_col_raw + [y_col_raw]].copy()
df_model[y_col_raw] = pd.to_numeric(df_model[y_col_raw], errors='coerce')
df_model = df_model.dropna(subset=[y_col_raw])

X_num = [c for c in X_col_raw if _isnumcol(c)]
X_cat = [c for c in X_col_raw if not _isnumcol(c)]
ct = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')),
                      ('scl', StandardScaler())]), X_num),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                      ('enc', OrdinalEncoder(handle_unknown='use_encoded_value',
                                             unknown_value=-1))]), X_cat),
], remainder='drop')

y_all      = df_model[y_col_raw].values.astype(float)
feat_names = [_lbl(c) for c in X_col_raw]

X_tr, X_te, y_tr, y_te = train_test_split(
    df_model[X_col_raw], y_all, test_size=test_size, random_state=rand_seed)
X_tr_t = ct.fit_transform(X_tr)
X_te_t = ct.transform(X_te)

if model_name == 'OLS (statsmodels)':
    _Xtr    = X_tr_t.toarray() if hasattr(X_tr_t, 'toarray') else X_tr_t
    _Xte    = X_te_t.toarray() if hasattr(X_te_t, 'toarray') else X_te_t
    _sm_fit = sm.OLS(y_tr, sm.add_constant(_Xtr)).fit()
    y_pred  = _sm_fit.predict(sm.add_constant(_Xte, has_constant='add'))
    _fitted = None
    display(HTML('<pre style="font-size:11px">' + _sm_fit.summary().as_text() + '</pre>'))
else:
    spec    = _CATALOG[model_name]
    _fitted = spec['cls'](**{**spec['fit_kw'], **_hp_vals()})
    _fitted.fit(X_tr_t, y_tr)
    y_pred  = _fitted.predict(X_te_t)

r2    = r2_score(y_te, y_pred)
n, p  = len(y_te), len(X_col_raw)
r2adj = 1 - (1 - r2) * (n - 1) / (n - p - 1) if n > p + 1 else float('nan')
rmse  = np.sqrt(mean_squared_error(y_te, y_pred))
mae   = mean_absolute_error(y_te, y_pred)
resid = y_te - y_pred

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
mn, mx = min(y_te.min(), y_pred.min()), max(y_te.max(), y_pred.max())
axes[0].scatter(y_te, y_pred, alpha=0.4, s=12, color='#6366f1')
axes[0].plot([mn, mx], [mn, mx], 'r--', lw=1.5)
axes[0].set_xlabel('Actual'); axes[0].set_ylabel('Predicted')
axes[0].set_title('Actual vs Predicted  (R2={:.3f})'.format(r2))
axes[1].scatter(y_pred, resid, alpha=0.4, s=12, color='#0ea5e9')
axes[1].axhline(0, color='red', lw=1.5, ls='--')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Residual')
axes[1].set_title('Residual plot')
axes[2].hist(resid, bins=30, color='#10b981', alpha=0.75, edgecolor='white')
axes[2].set_xlabel('Residual'); axes[2].set_ylabel('Count')
axes[2].set_title('Residual distribution')
plt.tight_layout(); plt.show()

top_note = ''
if _fitted is not None and hasattr(_fitted, 'feature_importances_'):
    top3 = [feat_names[i] for i in np.argsort(_fitted.feature_importances_)[::-1][:3]]
    top_note = '  Top features by impurity: **{}**, **{}**, **{}**.'.format(*top3)

if r2 >= 0.7:
    fit_note = 'R2 > 0.7 indicates good predictive fit.'
elif r2 < 0.3:
    fit_note = 'R2 below 0.3 -- the model explains little variance; try a more flexible model or additional features.'
else:
    fit_note = 'Moderate explanatory power.'

x_listed = ', '.join('**' + _lbl(c) + '**' for c in X_col_raw[:5])
x_listed += '...' if len(X_col_raw) > 5 else ''
printmd(
    '**' + model_name + ' results**\n\n'
    'R2 **{:.4f}** | Adj R2 **{:.4f}** | RMSE **{:.4g}** | MAE **{:.4g}**  \n'
    'Test set: {:,} samples ({:.0%} of {:,} total).{}\n\n{}'.format(
        r2, r2adj, rmse, mae,
        len(y_te), test_size, len(df_model), top_note, fit_note)
)


In [ ]:
# -- Step 4: SHAP feature importance ---------------------------------------
if model_name == 'OLS (statsmodels)' or _fitted is None:
    printmd('*OLS coefficients already provide direct interpretability with p-values '
            'and confidence intervals -- SHAP is not needed.*')
else:
    shap_type = _CATALOG[model_name]['shap']
    printmd('**SHAP feature importance ({} explainer):**'.format(shap_type))
    try:
        X_te_d = X_te_t.toarray() if hasattr(X_te_t, 'toarray') else X_te_t
        X_tr_d = X_tr_t.toarray() if hasattr(X_tr_t, 'toarray') else X_tr_t
        n_te   = min(200, len(X_te_d))
        if shap_type == 'tree':
            expl = shap.TreeExplainer(_fitted)
            sv   = expl.shap_values(X_te_d[:n_te])
        elif shap_type == 'linear':
            bg   = shap.maskers.Independent(X_tr_d, max_samples=100)
            expl = shap.LinearExplainer(_fitted, bg)
            sv   = expl.shap_values(X_te_d[:n_te])
        else:
            bg   = shap.kmeans(X_tr_d[:min(100, len(X_tr_d))], 10)
            expl = shap.KernelExplainer(_fitted.predict, bg)
            sv   = expl.shap_values(X_te_d[:50])

        sv2      = sv if hasattr(sv, 'ndim') else np.array(sv)
        mean_abs = np.abs(sv2).mean(axis=0)
        idx      = np.argsort(mean_abs)[::-1][:20]
        fig, ax  = plt.subplots(figsize=(8, max(4, len(idx) * 0.35 + 1)))
        ax.barh([feat_names[i] for i in idx[::-1]], mean_abs[idx[::-1]], color='#6366f1')
        ax.set_xlabel('Mean |SHAP value| -- average impact on predicted value')
        ax.set_title('SHAP feature importance -- ' + model_name)
        plt.tight_layout(); plt.show()

        top3 = [feat_names[i] for i in idx[:3]]
        printmd(
            'Features with greatest average influence: '
            '**{}**, **{}**, **{}**. '
            'SHAP values show how much each feature shifts the prediction '
            'away from the dataset mean.'.format(*top3)
        )
    except Exception as e:
        printmd('*SHAP computation failed: {}*'.format(e))


In [ ]:
# -- Step 5a: 5-fold cross-validation -------------------------------------
if model_name == 'OLS (statsmodels)' or _fitted is None:
    printmd('*Automated cross-validation is not available for statsmodels OLS here.*')
else:
    printmd('**5-fold cross-validation:**')
    cv       = KFold(n_splits=5, shuffle=True, random_state=rand_seed)
    X_full_t = ct.transform(df_model[X_col_raw])

    cv_r2   = cross_val_score(_fitted, X_full_t, y_all, cv=cv,
                               scoring='r2', n_jobs=-1)
    cv_rmse = np.sqrt(-cross_val_score(_fitted, X_full_t, y_all, cv=cv,
                                        scoring='neg_mean_squared_error', n_jobs=-1))

    x = np.arange(5)
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
    axes[0].bar(x, cv_r2,   color='#6366f1', alpha=0.85)
    axes[0].axhline(cv_r2.mean(), color='#6366f1', ls='--', lw=1.5)
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(['Fold {}'.format(i+1) for i in range(5)])
    axes[0].set_title('CV R2 per fold')
    axes[1].bar(x, cv_rmse, color='#0ea5e9', alpha=0.85)
    axes[1].axhline(cv_rmse.mean(), color='#0ea5e9', ls='--', lw=1.5)
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(['Fold {}'.format(i+1) for i in range(5)])
    axes[1].set_title('CV RMSE per fold')
    plt.tight_layout(); plt.show()

    if cv_r2.std() < 0.05:
        var_note = 'consistent -- good generalization'
    elif cv_r2.std() < 0.15:
        var_note = 'moderately variable'
    else:
        var_note = 'highly variable -- may indicate overfitting or small sample'

    printmd(
        'CV R2: **{:.4f} +/- {:.4f}** (5 folds)  \n'
        'CV RMSE: **{:.4g} +/- {:.4g}** (5 folds)  \n\n'
        'Generalization is **{}** (R2 std = {:.4f}).'.format(
            cv_r2.mean(), cv_r2.std(),
            cv_rmse.mean(), cv_rmse.std(),
            var_note, cv_r2.std())
    )


In [ ]:
# -- Optional: statsmodels OLS alongside an ML model ----------------------
if model_name == 'OLS (statsmodels)':
    printmd('*OLS summary already shown in the training cell -- skip this cell.*')
else:
    printmd('**Optional: OLS statistical summary for significance testing**')
    printmd('OLS provides p-values and confidence intervals that sklearn models do not. '
            'Use these for inference, not for prediction accuracy.')
    try:
        X_full_t = ct.transform(df_model[X_col_raw])
        _Xfull   = X_full_t.toarray() if hasattr(X_full_t, 'toarray') else X_full_t
        ols = sm.OLS(y_all, sm.add_constant(_Xfull)).fit()
        display(HTML('<pre style="font-size:11px">' + ols.summary().as_text() + '</pre>'))
    except Exception as e:
        printmd('*OLS could not be computed: {}*'.format(e))


In [ ]:
# -- Step 5b: save predictions and create new survey ----------------------
X_full_t = ct.transform(df_model[X_col_raw])
if model_name == 'OLS (statsmodels)':
    _Xfull    = X_full_t.toarray() if hasattr(X_full_t, 'toarray') else X_full_t
    all_preds = _sm_fit.predict(sm.add_constant(_Xfull, has_constant='add'))
else:
    all_preds = _fitted.predict(X_full_t)

pred_col = 'predicted_' + _lbl(y_col_raw) + '#number'
df_out   = df_raw.copy()
df_out.loc[df_model.index, pred_col] = np.round(all_preds, 6)

new_file    = suaveint.save_csv_file(df_out, absolutePath, csv_file)
survey_name = input('Enter name for new survey: ')
printmd('**Survey name:** ' + survey_name)
suaveint.create_survey(survey_url, new_file, survey_name, dzc_file, user, csv_file, view, views)
